# Commerce synthetic policy OCR comparison

Run one OCR engine against the same nine image-only Vietnamese policy pages used by the Commerce RAG demo. This is a paired OCR-engine experiment for a fictional business corpus, not real-company policy or physical-scan accuracy evidence. Ground truth is extracted from the authored HTML and has not been independently human-reviewed; do not use the aggregate score to select a production model until those labels are reviewed. Each run records source-PDF and rendered-image hashes, engine/package version, runtime device, initialization and inference time, process memory, CER/WER and any failures. Keep runs on the same Kaggle image size/settings for accuracy comparison; compare latency only on like hardware.


In [ ]:
from pathlib import Path
import shutil, subprocess, sys

input_root = Path("/kaggle/input")
bundle_files = sorted(input_root.rglob("project.zip")) + sorted(input_root.rglob("benchmark_project.zip"))
for bundle in bundle_files:
    extracted = Path("/kaggle/working/benchmark-assets/project")
    shutil.rmtree(extracted.parent, ignore_errors=True)
    shutil.unpack_archive(str(bundle), str(extracted.parent))
project_candidates = {
    *input_root.glob("*/project"),
    *input_root.glob("datasets/*/*/project"),
    *Path("/kaggle/working/benchmark-assets").glob("project"),
    *(entry.parent.parent.parent for entry in input_root.rglob("evaluate_ocr.py") if entry.parent.name == "processing" and entry.parent.parent.name == "scripts"),
}
assets_candidates = sorted(project for project in project_candidates if (project / "scripts/processing/evaluate_ocr.py").is_file())
if len(assets_candidates) != 1:
    raise FileNotFoundError(f"Expected one attached Commerce OCR dataset; found: {assets_candidates}")
assets = assets_candidates[0]
repo = Path("/kaggle/working/enterprise-knowledge-ai-agent")
if repo.exists():
    shutil.rmtree(repo)
shutil.copytree(assets, repo)
%cd /kaggle/working/enterprise-knowledge-ai-agent


In [ ]:
ENGINE = "tesseract"  # Run one engine per kernel: tesseract, paddleocr, easyocr, or surya.
REQUIRE_GPU = False  # Set True only for EasyOCR/Surya GPU runs.
DPI = 180
TESSERACT_PSM = 11
DATASET_PATH = "data/evaluation/ocr/commerce_policy_scan_ground_truth.json"
DATASET_TAG = "commerce-policy-scan-v1"


In [ ]:
import subprocess, sys

# Install one engine only, avoiding incompatible OCR framework combinations.
subprocess.run([sys.executable, "-m", "pip", "install", "PyMuPDF==1.27.2.3", "Pillow==12.1.0"], check=True)
if ENGINE == "paddleocr":
    subprocess.run([sys.executable, "-m", "pip", "--default-timeout=300", "--retries=5", "install", "paddlepaddle==3.3.0", "-i", "https://www.paddlepaddle.org.cn/packages/stable/cpu/"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "paddleocr==3.7.0"], check=True)
elif ENGINE == "easyocr":
    subprocess.run([sys.executable, "-m", "pip", "install", "easyocr==1.7.2"], check=True)
elif ENGINE == "surya":
    subprocess.run([sys.executable, "-m", "pip", "install", "surya-ocr==0.17.1", "transformers==4.57.3"], check=True)
elif ENGINE == "tesseract":
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "tesseract-ocr", "tesseract-ocr-vie"], check=True)
else:
    raise ValueError(f"Unknown ENGINE: {ENGINE}")


In [ ]:
import os

os.environ["HF_HOME"] = "/kaggle/working/model-cache/huggingface"
os.environ["TORCH_HOME"] = "/kaggle/working/model-cache/torch"
if ENGINE == "surya":
    # Fit Surya's detection/recognition batches within Kaggle's 15 GB Tesla T4.
    os.environ["DETECTOR_BATCH_SIZE"] = "16"
    os.environ["RECOGNITION_BATCH_SIZE"] = "128"
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
if ENGINE in {"easyocr", "surya"}:
    import torch
    cuda_available = torch.cuda.is_available()
    print("torch", torch.__version__, "CUDA", cuda_available)
    if REQUIRE_GPU and not cuda_available:
        raise RuntimeError("GPU requested for this OCR engine but CUDA is unavailable; restart with a Kaggle GPU accelerator.")
elif ENGINE == "paddleocr":
    import paddle
    print("paddle", paddle.__version__, "CUDA build", paddle.is_compiled_with_cuda(), "device", paddle.device.get_device())


In [ ]:
import json
dataset_path = Path(DATASET_PATH)
if not dataset_path.is_file():
    raise FileNotFoundError(f"Missing OCR labels: {dataset_path}")
dataset = json.loads(dataset_path.read_text(encoding="utf-8"))
for source in sorted({page["file"] for page in dataset["pages"]}):
    if not Path(source).is_file():
        raise FileNotFoundError(f"Missing benchmark source PDF: {source}")
output = Path("/kaggle/working") / f"commerce-ocr-{DATASET_TAG}-{ENGINE}.json"
command = [sys.executable, "scripts/processing/evaluate_ocr.py", "--dataset", DATASET_PATH, "--engines", ENGINE, "--dpi", str(DPI), "--output", str(output)]
if ENGINE == "tesseract":
    command.extend(["--tesseract-psm", str(TESSERACT_PSM)])
subprocess.run(command, check=True)
print(f"Download the result from Kaggle Outputs: {output}")


This run compares OCR implementations on authored, image-only synthetic Commerce policy PDFs. It does not establish production scan accuracy: references came from authored HTML and have not been independently human-reviewed, the PDFs are not physical scans, and this corpus has no tables. Compare only reports with matching dataset, source-PDF and rendered-image hashes. Record hardware when comparing speed; do not change the production upload engine from these results alone.